# Gameglory · ML / NLP Engineer · «Похожие игры»

Шаблон ноутбука. Заполните ячейки с пометкой **TODO**. Разделы 1–5 идут в том же порядке, что и пункты «Что сделать» в задании; пункт 6 (воспроизводимость) относится ко всему ноутбуку, последний раздел — к дополнительной части.

Загрузка данных и вспомогательные функции уже готовы: можно сразу переходить к разделу 1.

> Данные **синтетические**: названия и описания игр собраны из шаблонов. Учитывайте это в выводах.

Ноутбук должен выполняться сверху вниз на чистом окружении:
`jupyter nbconvert --to notebook --execute starter.ipynb --output executed.ipynb`

## 0. Загрузка данных (готово)

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Путь к shared-data: переменная окружения GAMEGLORY_DATA или ../shared-data относительно пакета
DATA = Path(os.environ.get("GAMEGLORY_DATA", "../shared-data")).resolve()
assert (DATA / "games.csv").exists(), f"Не найден games.csv в {DATA}; задайте GAMEGLORY_DATA"
print("DATA =", DATA)

games = pd.read_csv(DATA / "games.csv", parse_dates=["release_date"])
for col in ["tags", "genres", "platforms"]:
    # в CSV списки разделены символом '|'
    games[col] = games[col].fillna("").map(lambda s: [x for x in s.split("|") if x])
games["short_description"] = games["short_description"].fillna("")

pairs = pd.read_csv("similarity_eval_pairs.csv")

print(games.shape, pairs.shape)
games[["id", "title", "tags", "short_description"]].head(3)

In [ ]:
# Быстрые проверки данных
assert games["id"].is_unique
assert set(pairs["game_id_a"]).union(pairs["game_id_b"]) <= set(games["id"])
print("Тегов на игру:", games["tags"].map(len).describe().round(1).to_dict())
print("Длина описания, символов:", games["short_description"].str.len().describe().round(0).to_dict())
print("Метки пар:", pairs["label"].value_counts().to_dict())
print("Типы пар:", pairs["note"].value_counts().to_dict())

In [ ]:
# Вспомогательные функции (можно менять)
TITLE = dict(zip(games["id"], games["title"]))
TAGS = dict(zip(games["id"], games["tags"]))


def show(game_id: int, results: list[tuple[int, float]]) -> pd.DataFrame:
    """Таблица выдачи: название, score, теги — удобно смотреть глазами."""
    print(f"Запрос: {game_id} · {TITLE[game_id]} · {', '.join(TAGS[game_id])}")
    return pd.DataFrame(
        [{"game_id": gid, "title": TITLE[gid], "score": round(score, 3), "tags": ", ".join(TAGS[gid])}
         for gid, score in results]
    )


def score_pairs(score_fn) -> np.ndarray:
    """score_fn(a, b) -> float для каждой из 50 пар similarity_eval_pairs.csv."""
    return np.array([score_fn(a, b) for a, b in zip(pairs["game_id_a"], pairs["game_id_b"])])

## 1. Поиск по тегам — Жаккар

**TODO.** `jaccard(a, b)` и `similar_by_tags(game_id, k=10) -> list[(game_id, score)]`, отсортированный по убыванию score. Сама игра в выдачу не входит.

In [ ]:
def jaccard(a: int, b: int) -> float:
    # TODO: |A ∩ B| / |A ∪ B| по множествам тегов
    return 0.0


def similar_by_tags(game_id: int, k: int = 10) -> list[tuple[int, float]]:
    # TODO
    return []

## 2. Поиск по описаниям — TF-IDF

**TODO.** TF-IDF по `short_description` + косинусное сходство. Кратко объясните выбранные параметры `TfidfVectorizer` (стоп-слова, n-граммы, `min_df` и т. п.).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO


def tfidf_sim(a: int, b: int) -> float:
    # TODO: косинус между TF-IDF-векторами описаний
    return 0.0


def similar_by_text(game_id: int, k: int = 10) -> list[tuple[int, float]]:
    # TODO
    return []

## 3. Демонстрация

**TODO.** Выберите 5 игр из **разных** ниш (колонки «ниша» в данных нет: определите её по тегам и объясните выбор) и покажите top-10 обоих подходов. Для каждой игры — 2–3 предложения: где выдача удачная, где ошибается и почему.

In [ ]:
DEMO_IDS: list[int] = []  # TODO: 5 game_id

for gid in DEMO_IDS:
    display(show(gid, similar_by_tags(gid)))
    display(show(gid, similar_by_text(gid)))

**Комментарии к выдачам:** TODO

## 4. Оценка на 50 размеченных парах

`similarity_eval_pairs.csv`: 25 пар «похожи» (`label = 1`) и 25 «не похожи» (`label = 0`); 10 из отрицательных — «трудные» (игры из соседних ниш, см. колонку `note`).

**TODO.** Для каждого подхода: score каждой пары → ROC-AUC. Если считаете accuracy/F1 при пороге — объясните, как выбран порог (не подбирайте его на этих же 50 парах без оговорки). Разберите 2–3 худшие ошибки каждого метода: похожие пары с самым низким score и непохожие с самым высоким.

In [ ]:
from sklearn.metrics import roc_auc_score

pairs["jaccard"] = score_pairs(jaccard)
pairs["tfidf"] = score_pairs(tfidf_sim)

# TODO: ROC-AUC для каждого подхода и разбор ошибок


## 5. Сравнение и выводы

**TODO.** Таблица «подход × метрика». Какой подход и для какого сценария вы бы выбрали? Минимум 3 конкретных ограничения (размер выборки, синтетичность и шаблонность текстов, что изменится на реальных данных Steam). Что бы сделали дальше.

In [ ]:
# TODO: сводная таблица
summary = pd.DataFrame(columns=["method", "roc_auc"])
summary

**Выводы и ограничения:** TODO

## Дополнительно (по желанию)

Варианты — как в разделе «Дополнительно» задания: эмбеддинги, bootstrap-интервалы или гибрид, метрика без разметки, темы отзывов (`reviews.csv`). Для эмбеддингов понадобятся дополнительные библиотеки — добавьте их в `requirements.txt`.

In [ ]:
# по желанию
